# Polymarket data inventory and bounded raw-data sample

This notebook explores the shared manifest **without downloading the snapshot**. It then downloads only (1) one daily market index and (2) a contiguous, latest-day raw window. A hard 1 GiB compressed-byte budget applies to both downloads together.

Snapshot coverage: 2026-05-23 through 2026-08-16. Signed URLs expire 2026-09-28 13:17 Asia/Shanghai.

In [ ]:
from __future__ import annotations

import io
import json
import re
import shutil
import subprocess
import urllib.request
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
# If launched from a different directory, change this one value.
SHARE_ROOT = PROJECT_ROOT / 'poly-data-share' / 'poly-data-share'
MANIFEST = SHARE_ROOT / 'manifest.tsv'
DOWNLOAD_ROOT = PROJECT_ROOT / 'data' / 'exploration_sample'
MAX_TOTAL_BYTES = 1024**3  # Strict cap: index + raw files must stay below 1 GiB.

assert MANIFEST.exists(), f'Manifest not found: {MANIFEST.resolve()}'
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Manifest: {MANIFEST.resolve()}')
print(f'Sample destination: {DOWNLOAD_ROOT.resolve()}')

In [ ]:
manifest = pd.read_csv(MANIFEST, sep='\t', header=None, names=['bytes', 'path', 'url'])
manifest['bytes'] = manifest['bytes'].astype('int64')
manifest['gib'] = manifest['bytes'] / 1024**3
manifest['top_level'] = manifest['path'].str.split('/').str[:2].str.join('/')
manifest['date'] = manifest['path'].str.extract(r'^raw/(\d{4}-\d{2}-\d{2})/')[0]

inventory = (manifest.groupby('top_level', as_index=False)
             .agg(objects=('path', 'size'), gib=('gib', 'sum'))
             .sort_values('gib', ascending=False))
print(f"{len(manifest):,} objects; {manifest['gib'].sum():,.3f} GiB ({manifest['gib'].sum() / 1024:.3f} TiB)")
display(inventory.assign(gib=inventory['gib'].round(2)))

primary = manifest[manifest['date'].notna()]
daily_primary = (primary.groupby('date', as_index=False)
                 .agg(hourly_objects=('path', 'size'), gib=('gib', 'sum'))
                 .sort_values('date'))
display(daily_primary.tail(10).assign(gib=lambda x: x['gib'].round(2)))

## Market catalogue

The `polymarket_index.json` object is a compact market catalogue, so it is downloaded before any raw feed. The newest one is selected automatically.

In [ ]:
def safe_download(entry: pd.Series, *, budget_remaining: int) -> Path:
    """Download exactly one manifest object, refusing to exceed its allocated budget."""
    expected = int(entry.bytes)
    if expected > budget_remaining:
        raise ValueError(f"Refusing {entry.path}: {expected / 1024**2:.1f} MiB exceeds remaining budget")
    destination = DOWNLOAD_ROOT / Path(entry.path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size == expected:
        print(f'Using cached {entry.path} ({expected / 1024**2:.1f} MiB)')
        return destination
    partial = destination.with_suffix(destination.suffix + '.part')
    offset = partial.stat().st_size if partial.exists() else 0
    if offset > expected:
        partial.unlink()
        offset = 0
    # Ranged requests make partial downloads resumable and avoid asking the server for an unbounded body.
    chunk_size = 8 * 1024**2
    while offset < expected:
        end = min(offset + chunk_size - 1, expected - 1)
        request = urllib.request.Request(entry.url, headers={'Range': f'bytes={offset}-{end}'})
        with urllib.request.urlopen(request, timeout=120) as response:
            chunk = response.read()
        wanted = end - offset + 1
        if len(chunk) != wanted or offset + len(chunk) > budget_remaining:
            raise IOError(f'Unexpected ranged response for {entry.path}')
        with partial.open('ab') as out:
            out.write(chunk)
        offset += len(chunk)
        print(f'  {offset / expected:.0%}', end='\r')
    print()
    if partial.stat().st_size != expected:
        raise IOError(f"Size mismatch for {entry.path}: expected {expected}, got {partial.stat().st_size}")
    partial.replace(destination)
    print(f'Downloaded {entry.path} ({expected / 1024**2:.1f} MiB)')
    return destination

index_candidates = manifest[manifest['path'].str.match(r'^raw/_index/\d{4}-\d{2}-\d{2}/polymarket_index\.json$')].copy()
index_candidates['index_date'] = index_candidates['path'].str.extract(r'^raw/_index/(\d{4}-\d{2}-\d{2})/')[0]
index_entry = index_candidates.sort_values('index_date').iloc[-1]
print(f"Selected market index: {index_entry.path} ({index_entry.bytes / 1024**2:.1f} MiB)")
index_path = safe_download(index_entry, budget_remaining=MAX_TOTAL_BYTES)
index_payload = json.loads(index_path.read_text(encoding='utf-8'))
print(f'Index root type: {type(index_payload).__name__}')
print(f'Root keys: {list(index_payload)[:20]}' if isinstance(index_payload, dict) else f'Root records: {len(index_payload):,}')

In [ ]:
def index_to_market_frame(payload):
    """Normalize the known market index, retaining its market id dictionary key."""
    if isinstance(payload, dict) and isinstance(payload.get('market_index'), dict):
        frame = pd.DataFrame.from_dict(payload['market_index'], orient='index')
        frame.index.name = 'index_market_id'
        print(f"Using payload['market_index'] with {len(frame):,} market records")
        return frame.reset_index()
    if isinstance(payload, list):
        return pd.json_normalize(payload, sep='.')
    raise ValueError("Expected a 'market_index' dictionary or list payload; inspect index_payload above.")

markets = index_to_market_frame(index_payload)
print(f'{len(markets):,} index records; {len(markets.columns):,} fields')
display(pd.DataFrame({'column': markets.columns, 'non_null': markets.notna().sum().to_numpy()}).head(80))

# Display likely identifiers and human-readable market labels without assuming a fixed schema.
likely = [c for c in markets.columns if any(token in c.lower() for token in ('id', 'slug', 'title', 'question', 'market', 'event', 'category', 'tag'))]
display(markets[likely[:20]].head(25))

# Initial concrete choice: BNB Up or Down — August 16, 12PM ET (market id from this index).
# Change this after reviewing the table; it is only used in the optional matching cell below.
MARKET_QUERY = '3616667'  # or use a title fragment / an exact market, event, or token id
market_matches = markets[markets.astype(str).apply(lambda col: col.str.contains(MARKET_QUERY, case=False, na=False)).any(axis=1)].copy()
# Raw feed messages use the 32-byte condition id as a 0x-prefixed hex string.
market_matches['raw_market_id'] = market_matches['index_market_id'].map(lambda value: f'0x{int(value):064x}')
MARKET_RAW_IDS = set(market_matches['raw_market_id'])
print(f"Catalogue matches for {MARKET_QUERY!r}: {len(market_matches):,}")
display(market_matches)

## Bounded tick-level raw sample

Raw objects are hourly `.jsonl.zst` files containing all markets, not one object per market. The cell below selects the **15:00–17:00 UTC two-hour run on the latest date**, which includes the selected 12PM ET BNB contract. It intentionally excludes retry/suffixed files and never expands the selection beyond the cap.

In [ ]:
remaining = MAX_TOTAL_BYTES - int(index_entry.bytes)
MAX_RAW_HOURS = 2  # A useful first pass; increase only after inspecting the sampled schema.
SAMPLE_START_HOUR_UTC = 15  # 11AM ET; covers the selected BNB contract through its 1PM ET close.
hourly = manifest[manifest['path'].str.match(r'^raw/\d{4}-\d{2}-\d{2}/[0-2]\d00\.jsonl\.zst$')].copy()
hourly['date'] = hourly['path'].str.extract(r'^raw/(\d{4}-\d{2}-\d{2})/')[0]
hourly['hour'] = hourly['path'].str.extract(r'/([0-2]\d)00\.jsonl')[0].astype(int)
latest_date = hourly['date'].max()
day = hourly[hourly['date'].eq(latest_date) & hourly['hour'].ge(SAMPLE_START_HOUR_UTC)].sort_values('hour').reset_index(drop=True)

# A sliding window ensures both contiguity and the remaining compressed-byte budget.
best = None
for start in range(len(day)):
    total = 0
    for end in range(start, len(day)):
        if end > start and day.loc[end, 'hour'] != day.loc[end - 1, 'hour'] + 1:
            break
        total += int(day.loc[end, 'bytes'])
        if total > remaining:
            break
        candidate = (end - start + 1, total, start, end)
        if best is None or candidate[:2] > best[:2]:
            best = candidate
        if candidate[0] >= MAX_RAW_HOURS:
            break
assert best is not None, 'No latest-day raw file fits the remaining budget.'
_, raw_bytes, start, end = best
raw_window = day.loc[start:end].copy()
print(f"Raw window: {latest_date} {raw_window.hour.iloc[0]:02d}:00–{raw_window.hour.iloc[-1] + 1:02d}:00 UTC")
print(f"Coverage: {len(raw_window)} contiguous hour(s); compressed download: {raw_bytes / 1024**2:.1f} MiB")
print(f"Total planned download (index + raw): {(raw_bytes + int(index_entry.bytes)) / 1024**2:.1f} MiB / 1024.0 MiB")
assert raw_bytes + int(index_entry.bytes) <= MAX_TOTAL_BYTES
display(raw_window[['path', 'bytes']].assign(mib=lambda x: x.bytes / 1024**2))

# This downloads only the rows displayed above; rerunning is resumable through the local cache.
raw_paths = []
budget_left = remaining
for _, entry in raw_window.iterrows():
    raw_paths.append(safe_download(entry, budget_remaining=budget_left))
    budget_left -= int(entry.bytes)

In [ ]:
# Stream a small prefix; do not decompress the full sample into memory or write an uncompressed copy.
ZSTD = shutil.which('zstd')
assert ZSTD, 'zstd executable not found. Install zstd or add it to PATH before running this cell.'

def jsonl_prefix_zstd(path: Path, limit: int = 25):
    process = subprocess.Popen([ZSTD, '-dc', '--', str(path)], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, encoding='utf-8', errors='replace')
    try:
        rows = []
        for line in process.stdout:
            if line.strip():
                rows.append(json.loads(line))
            if len(rows) >= limit:
                break
        return rows
    finally:
        process.stdout.close()
        process.terminate()
        process.wait()

raw_preview_records = jsonl_prefix_zstd(raw_paths[0])
raw_preview = pd.json_normalize(raw_preview_records, sep='.')
def parse_content(value):
    try:
        return json.loads(value) if isinstance(value, str) else value
    except (TypeError, json.JSONDecodeError):
        return {}
raw_preview['content_json'] = raw_preview['content'].map(parse_content)
raw_preview['raw_market_id'] = raw_preview['content_json'].map(lambda value: value.get('market'))
print(f'Previewed {len(raw_preview):,} JSON records from {raw_paths[0].name}; {len(raw_preview.columns):,} fields')
display(pd.DataFrame({'column': raw_preview.columns, 'non_null': raw_preview.notna().sum().to_numpy()}).head(100))
display(raw_preview.head(10))

# The market catalogue's condition ids match the raw feed's `content_json.market` field.
hits = raw_preview['raw_market_id'].isin(MARKET_RAW_IDS)
print(f'Matching selected-market messages in this {len(raw_preview):,}-record prefix: {hits.sum():,}')
display(raw_preview.loc[hits, ['timestamp', 'message_type', 'raw_market_id', 'content_json']])

## Flatten nested feed messages

The outer JSONL record has an ingest timestamp and message type; its `content` field is another JSON document. The flattener below emits one row per price change (or one row per book level for book messages), retaining `market_id`, both timestamps, and event type. It streams zstd input and caps output rows, so it does not inflate a whole raw hour in memory.

In [ ]:
import io
# `zstandard` permits tracking compressed bytes consumed, which gives tqdm a real ETA.
# Installed for this project with: python -m pip install zstandard
import zstandard as zstd
from tqdm.auto import tqdm

class ProgressReader(io.RawIOBase):
    def __init__(self, raw, *bars):
        self.raw, self.bars = raw, bars
    def readable(self):
        return True
    def readinto(self, buffer):
        count = self.raw.readinto(buffer)
        if count:
            for bar in self.bars:
                bar.update(count)
        return count

def iter_jsonl_zstd(path: Path, overall_progress):
    file_progress = tqdm(total=path.stat().st_size, unit='B', unit_scale=True, desc=path.name, leave=True)
    compressed = path.open('rb')
    tracked = ProgressReader(compressed, file_progress, overall_progress)
    reader = zstd.ZstdDecompressor().stream_reader(tracked)
    text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
    try:
        for line in text_stream:
            if line.strip():
                yield json.loads(line)
    finally:
        text_stream.close()
        reader.close()
        compressed.close()
        file_progress.close()

def scalar_market_id(content: dict):
    """Some non-feed payloads use a nested `market` object; only IDs are joinable/filterable."""
    value = content.get('market') if isinstance(content, dict) else None
    return value if isinstance(value, (str, int)) else None

def flatten_message(record: dict):
    content = parse_content(record.get('content'))
    if not isinstance(content, dict):
        content = {}
    base = {
        'ingest_time': record.get('timestamp'),
        'message_type': record.get('message_type'),
        'market_id': scalar_market_id(content),  # 0x-prefixed condition id; joins to index_market_id after hex conversion
        'event_type': content.get('event_type'),
        'event_timestamp_ms': content.get('timestamp'),
    }
    # Price changes carry one or more independent ticks in the same feed message.
    if isinstance(content.get('price_changes'), list):
        for change in content['price_changes']:
            yield {**base, 'row_kind': 'price_change', **change}
        return
    # A book snapshot can carry many levels. Flatten them with an explicit side.
    emitted_level = False
    for side in ('bids', 'asks'):
        levels = content.get(side)
        if isinstance(levels, list):
            for level in levels:
                if isinstance(level, dict):
                    yield {**base, 'row_kind': 'book_level', 'book_side': side[:-1].upper(), **level}
                elif isinstance(level, (list, tuple)) and len(level) >= 2:
                    yield {**base, 'row_kind': 'book_level', 'book_side': side[:-1].upper(), 'price': level[0], 'size': level[1]}
                emitted_level = True
    if emitted_level:
        return
    # Preserve scalar fields for event types that are neither price changes nor book snapshots.
    details = {key: value for key, value in content.items()
               if key not in {'market', 'event_type', 'timestamp', 'price_changes', 'bids', 'asks'}
               and not isinstance(value, (dict, list))}
    yield {**base, 'row_kind': 'event', **details}

def flattened_chunks(paths, *, market_ids=None, batch_rows=50_000, max_rows=200_000, scan_stats=None):
    """Yield bounded, tidy DataFrames; the progress bars report compressed-byte ETA and output rows."""
    wanted = set(market_ids) if market_ids else None
    total_bytes = sum(path.stat().st_size for path in paths)
    buffer, emitted = [], 0
    with tqdm(total=total_bytes, unit='B', unit_scale=True, desc='All selected files', leave=True) as overall, \
         tqdm(total=max_rows, unit='rows', desc='Flattened rows', leave=True) as row_progress:
        for path in paths:
            for record in iter_jsonl_zstd(path, overall):
                if scan_stats is not None:
                    scan_stats['records_scanned'] = scan_stats.get('records_scanned', 0) + 1
                content = parse_content(record.get('content'))
                raw_market = scalar_market_id(content)
                if raw_market and scan_stats is not None:
                    scan_stats['market_messages'] = scan_stats.get('market_messages', 0) + 1
                if wanted and raw_market not in wanted:
                    continue
                for row in flatten_message(record):
                    if emitted >= max_rows:
                        return
                    buffer.append(row)
                    emitted += 1
                    row_progress.update(1)
                    if len(buffer) == batch_rows:
                        yield pd.DataFrame(buffer)
                        buffer = []
        if buffer:
            yield pd.DataFrame(buffer)

# The index is a daily active-market snapshot, not a guarantee that a condition id occurs in each raw hour.
# Start by discovering actual IDs in the raw feed. Flip this to True only after confirming the selected IDs occur.
FILTER_TO_SELECTED_MARKET = False
MAX_FLAT_ROWS = 50_000
market_filter = MARKET_RAW_IDS if FILTER_TO_SELECTED_MARKET else None
scan_stats = {}
flat_parts = list(flattened_chunks(raw_paths, market_ids=market_filter, max_rows=MAX_FLAT_ROWS, scan_stats=scan_stats))
flat_df = pd.concat(flat_parts, ignore_index=True) if flat_parts else pd.DataFrame()
if not flat_df.empty:
    flat_df['ingest_time'] = pd.to_datetime(flat_df['ingest_time'], utc=True, errors='coerce')
    flat_df['event_time'] = pd.to_datetime(flat_df['event_timestamp_ms'], unit='ms', utc=True, errors='coerce')
    flat_df = flat_df.sort_values(['market_id', 'event_time', 'event_type'], kind='stable').reset_index(drop=True)
print(f"Scanned {scan_stats.get('records_scanned', 0):,} outer records; saw {scan_stats.get('market_messages', 0):,} messages with scalar market IDs.")
print(f'Flattened {len(flat_df):,} rows (cap: {MAX_FLAT_ROWS:,}).')
if flat_df.empty and FILTER_TO_SELECTED_MARKET:
    print('No selected-market messages occurred in these files. Leave FILTER_TO_SELECTED_MARKET=False to discover available raw markets.')
else:
    display(flat_df.head(20))
    market_summary = (flat_df.groupby('market_id', dropna=True)
                      .agg(rows=('market_id', 'size'), first_event=('event_time', 'min'), last_event=('event_time', 'max'), event_types=('event_type', lambda x: ', '.join(sorted(x.dropna().unique()))))
                      .sort_values(['rows', 'last_event'], ascending=[False, False]))
    display(market_summary.head(25))
# Optional durable output after inspecting the schema: flat_df.to_parquet(DOWNLOAD_ROOT / 'bnb_2026-08-16_1500_1600_flat.parquet', index=False)